# Dynasty Utility Calculator

Clean rebuild — contains only the path from raw data pull to the final combined top-250 dynasty utility ranking. Every step here was validated against real output before being kept in.

## Stage 1: Sample — load raw data

In [ ]:
!pip install nflreadpy -q

import nflreadpy as nfl
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.inspection import permutation_importance

print("Libraries loaded")

In [ ]:
def load_weekly_stats(start_year: int, end_year: int) -> pd.DataFrame:
    """Week-by-week player stats (targets, yards, EPA, etc.) for every year in range."""
    years = list(range(start_year, end_year + 1))
    return nfl.load_player_stats(seasons=years).to_pandas()


def load_rosters(start_year: int, end_year: int) -> pd.DataFrame:
    """Player bio info -- birth date, years in league, position, team."""
    years = list(range(start_year, end_year + 1))
    return nfl.load_rosters(seasons=years).to_pandas()


def load_next_gen_stats(start_year: int, end_year: int) -> dict:
    """Advanced tracking stats (separation, time to throw, efficiency, etc.)."""
    years = list(range(start_year, end_year + 1))
    return {
        'passing': nfl.load_nextgen_stats(seasons=years, stat_type='passing').to_pandas(),
        'receiving': nfl.load_nextgen_stats(seasons=years, stat_type='receiving').to_pandas(),
        'rushing': nfl.load_nextgen_stats(seasons=years, stat_type='rushing').to_pandas(),
    }

print("Load functions defined")

In [ ]:
# Primary window used for the skill/opportunity models
START_YEAR = 2023
END_YEAR = 2025

weekly = load_weekly_stats(START_YEAR, END_YEAR)
rosters = load_rosters(START_YEAR, END_YEAR)
ngs = load_next_gen_stats(START_YEAR, END_YEAR)

print(f"Weekly stats: {weekly.shape}")
print(f"Rosters: {rosters.shape}")
print(f"NGS passing: {ngs['passing'].shape}")
print(f"NGS receiving: {ngs['receiving'].shape}")
print(f"NGS rushing: {ngs['rushing'].shape}")

## Stage 3: Modify — build `age_WearNTear`

Blends age (biological decline) and cumulative touches (wear), z-scored within position, with position-specific weights (RB skews wear-heavy, QB skews age-heavy).

In [ ]:
def approx_game_date(row):
    season_start = pd.Timestamp(year=int(row['season']), month=9, day=5)
    return season_start + timedelta(weeks=int(row['week']) - 1)


def zscore_by_position(s, pos_series):
    return s.groupby(pos_series).transform(lambda x: (x - x.mean()) / x.std())


# position-specific weights: (age_weight, wear_weight)
weights = {
    'QB': (0.75, 0.25),
    'RB': (0.35, 0.65),
    'WR': (0.55, 0.45),
    'TE': (0.55, 0.45),
}

print("Aging helper functions + weights defined")

In [ ]:
def build_age_weartear(weekly_df: pd.DataFrame, rosters_df: pd.DataFrame) -> pd.DataFrame:
    """Merges bio data onto weekly stats and engineers age_WearNTear."""
    roster_bio = rosters_df[['gsis_id', 'birth_date']].drop_duplicates(subset='gsis_id')
    df = weekly_df.merge(roster_bio, left_on='player_id', right_on='gsis_id', how='left')

    df['game_date'] = df.apply(approx_game_date, axis=1)
    df['birth_date'] = pd.to_datetime(df['birth_date'])
    df['age_years'] = (df['game_date'] - df['birth_date']).dt.days / 365.25

    df['carries'] = df['carries'].fillna(0)
    df['attempts'] = df['attempts'].fillna(0)
    df['targets'] = df['targets'].fillna(0)

    df['touches'] = np.where(
        df['position'] == 'QB',
        df['attempts'] + df['carries'],
        df['carries'] + df['targets']
    )

    df = df.sort_values(['player_id', 'season', 'week'])
    df['cumulative_touches'] = df.groupby('player_id')['touches'].cumsum()

    df['age_z'] = zscore_by_position(df['age_years'], df['position'])
    df['wear_z'] = zscore_by_position(df['cumulative_touches'], df['position'])

    df['age_weight'] = df['position'].map(lambda p: weights.get(p, (0.5, 0.5))[0])
    df['wear_weight'] = df['position'].map(lambda p: weights.get(p, (0.5, 0.5))[1])

    df['age_WearNTear'] = df['age_weight'] * df['age_z'] + df['wear_weight'] * df['wear_z']
    return df


weekly_wnt = build_age_weartear(weekly, rosters)
print(f"weekly_wnt shape: {weekly_wnt.shape}")
print("Missing age_WearNTear (no birth_date match):", weekly_wnt['age_WearNTear'].isna().sum())

## Build `base` — the working feature table

Opportunity, efficiency, aging, and NGS features merged at player-week grain (2023-2025).

In [ ]:
id_cols = ['player_id', 'player_display_name', 'position', 'season', 'week']

opportunity_cols = ['carries', 'attempts', 'targets', 'receptions',
                     'target_share', 'air_yards_share', 'wopr',
                     'touches', 'cumulative_touches']

efficiency_cols = ['passing_epa', 'rushing_epa', 'receiving_epa',
                    'passing_cpoe', 'racr', 'receiving_yards_after_catch']

aging_cols = ['age_years', 'age_z', 'wear_z', 'age_WearNTear']

target_col = ['fantasy_points_ppr']

base = weekly_wnt[id_cols + opportunity_cols + efficiency_cols + aging_cols + target_col].copy()

# NGS passing
ngs_pass_cols = ['player_gsis_id', 'season', 'week',
                  'avg_time_to_throw', 'completion_percentage_above_expectation', 'aggressiveness']
base = base.merge(
    ngs['passing'][ngs_pass_cols],
    left_on=['player_id', 'season', 'week'], right_on=['player_gsis_id', 'season', 'week'], how='left'
).drop(columns='player_gsis_id')

# NGS receiving
ngs_rec_cols = ['player_gsis_id', 'season', 'week',
                 'avg_separation', 'avg_cushion', 'avg_yac_above_expectation']
base = base.merge(
    ngs['receiving'][ngs_rec_cols],
    left_on=['player_id', 'season', 'week'], right_on=['player_gsis_id', 'season', 'week'], how='left'
).drop(columns='player_gsis_id')

# NGS rushing
ngs_rush_cols = ['player_gsis_id', 'season', 'week',
                  'rush_yards_over_expected_per_att', 'efficiency', 'percent_attempts_gte_eight_defenders']
base = base.merge(
    ngs['rushing'][ngs_rush_cols],
    left_on=['player_id', 'season', 'week'], right_on=['player_gsis_id', 'season', 'week'], how='left',
    suffixes=('', '_rush')
).drop(columns='player_gsis_id')

print(f"base shape: {base.shape}")

## Aging / Retention Curve

Built on a wider 2015-2025 window (for sample size only -- the skill models above stay on 2023-2025). Produces `aging_curve_lookup`: a `retention_multiplier` and `exit_rate` per position x wear-tercile.

In [ ]:
HIST_START_YEAR = 2015
HIST_END_YEAR = 2025

weekly_hist = load_weekly_stats(HIST_START_YEAR, HIST_END_YEAR)
rosters_hist = load_rosters(HIST_START_YEAR, HIST_END_YEAR)
weekly_hist_wnt = build_age_weartear(weekly_hist, rosters_hist)

print(f"weekly_hist_wnt shape: {weekly_hist_wnt.shape}")

In [ ]:
# player-season aggregation + next-season points (for retention ratio)
season_agg_hist = weekly_hist_wnt.groupby(['player_id', 'player_display_name', 'position', 'season']).agg(
    fantasy_points_ppr=('fantasy_points_ppr', 'sum'),
    games_played=('week', 'nunique'),
    age_years=('age_years', 'mean'),
    age_WearNTear=('age_WearNTear', 'mean'),
    cumulative_touches=('cumulative_touches', 'max'),
).reset_index()

season_agg_hist_sorted = season_agg_hist.sort_values(['player_id', 'season'])
next_season_hist = season_agg_hist_sorted[['player_id', 'season', 'fantasy_points_ppr']].copy()
next_season_hist['season'] = next_season_hist['season'] - 1
next_season_hist = next_season_hist.rename(columns={'fantasy_points_ppr': 'next_season_pts'})

season_agg_hist_sorted = season_agg_hist_sorted.merge(next_season_hist, on=['player_id', 'season'], how='left')
season_agg_hist_sorted['retention_ratio'] = (
    season_agg_hist_sorted['next_season_pts'] / season_agg_hist_sorted['fantasy_points_ppr']
)
season_agg_hist_sorted['exited'] = season_agg_hist_sorted['next_season_pts'].isna()

print(f"Player-seasons (historical): {len(season_agg_hist_sorted)}")

In [ ]:
def safe_qcut(x):
    try:
        return pd.qcut(x, 3, labels=['Low', 'Med', 'High'], duplicates='drop')
    except ValueError:
        return pd.qcut(x, 3, labels=False, duplicates='drop')


GAMES_FLOOR = 6

# qualifying filter -- keeps exited players too, needed for exit_rate
valid_hist_full = season_agg_hist_sorted[
    (season_agg_hist_sorted['fantasy_points_ppr'] >= 50) &
    (season_agg_hist_sorted['games_played'] >= GAMES_FLOOR) &
    (season_agg_hist_sorted['position'].isin(['QB', 'RB', 'WR', 'TE']))
].copy()

valid_hist_full['wnt_bucket'] = valid_hist_full.groupby('position')['age_WearNTear'].transform(safe_qcut)

retained_only = valid_hist_full[~valid_hist_full['exited']]

retention_stats = retained_only.groupby(['position', 'wnt_bucket'], observed=True).agg(
    median_retention=('retention_ratio', 'median'),
    mean_retention=('retention_ratio', 'mean'),
    n_retained=('retention_ratio', 'count'),
).reset_index()

exit_stats = valid_hist_full.groupby(['position', 'wnt_bucket'], observed=True).agg(
    n_total=('exited', 'count'),
    n_exited=('exited', 'sum'),
).reset_index()
exit_stats['exit_rate'] = (exit_stats['n_exited'] / exit_stats['n_total']).round(3)

aging_curve_final = retention_stats.merge(exit_stats, on=['position', 'wnt_bucket'])
aging_curve_lookup = aging_curve_final[['position', 'wnt_bucket', 'median_retention', 'exit_rate']].rename(
    columns={'median_retention': 'retention_multiplier'}
)

print("Aging curve lookup table:")
aging_curve_lookup

In [ ]:
# absolute-age retirement-cliff factor
# empirical per-age-per-position exit rates were too noisy past ~32 (n often <10) to trust directly,
# so this uses the aggregate 33+ exit rate measured across all positions as a flat rule
AGE_CLIFF_THRESHOLD = 33
AGE_CLIFF_DISCOUNT = 0.29

def age_cliff_factor(age):
    return (1 - AGE_CLIFF_DISCOUNT) if age >= AGE_CLIFF_THRESHOLD else 1.0

print("Age cliff rule defined: 33+ gets a flat", AGE_CLIFF_DISCOUNT, "discount")

In [ ]:
# exact tercile edges from the historical build -- needed to bucket NEW player-seasons consistently
bucket_edges = {}
for pos in ['QB', 'RB', 'WR', 'TE']:
    pos_data = valid_hist_full[valid_hist_full['position'] == pos]['age_WearNTear']
    _, edges = pd.qcut(pos_data, 3, retbins=True, duplicates='drop')
    bucket_edges[pos] = edges


def assign_bucket(wnt_value, pos):
    edges = bucket_edges[pos]
    labels = ['Low', 'Med', 'High']
    for i in range(len(edges) - 1):
        if wnt_value <= edges[i + 1]:
            return labels[i]
    return labels[-1]

print("Bucket edges captured for:", list(bucket_edges.keys()))

## Skill Models — WR / TE

Predicts raw season `fantasy_points_ppr` from opportunity-share + efficiency + NGS features (no raw counting stats, so the model can't just rediscover the scoring formula).

In [ ]:
SEED = 474
KFOLDS = 10

season_features = base.groupby(['player_id', 'player_display_name', 'position', 'season']).agg(
    carries=('carries', 'sum'), attempts=('attempts', 'sum'), targets=('targets', 'sum'),
    receptions=('receptions', 'sum'), target_share=('target_share', 'mean'),
    air_yards_share=('air_yards_share', 'mean'), wopr=('wopr', 'mean'),
    touches=('touches', 'sum'), cumulative_touches=('cumulative_touches', 'max'),
    passing_epa=('passing_epa', 'mean'), rushing_epa=('rushing_epa', 'mean'),
    receiving_epa=('receiving_epa', 'mean'), passing_cpoe=('passing_cpoe', 'mean'),
    racr=('racr', 'mean'), receiving_yards_after_catch=('receiving_yards_after_catch', 'sum'),
    avg_time_to_throw=('avg_time_to_throw', 'mean'),
    completion_percentage_above_expectation=('completion_percentage_above_expectation', 'mean'),
    aggressiveness=('aggressiveness', 'mean'), avg_separation=('avg_separation', 'mean'),
    avg_cushion=('avg_cushion', 'mean'), avg_yac_above_expectation=('avg_yac_above_expectation', 'mean'),
    rush_yards_over_expected_per_att=('rush_yards_over_expected_per_att', 'mean'),
    efficiency=('efficiency', 'mean'),
    percent_attempts_gte_eight_defenders=('percent_attempts_gte_eight_defenders', 'mean'),
    games_played=('week', 'nunique'), fantasy_points_ppr=('fantasy_points_ppr', 'sum'),
).reset_index()

position_features_skill = {
    'WR': ['target_share', 'air_yards_share', 'wopr', 'receiving_epa', 'racr',
           'avg_separation', 'avg_cushion', 'avg_yac_above_expectation'],
    'TE': ['target_share', 'air_yards_share', 'wopr', 'receiving_epa', 'racr',
           'avg_separation', 'avg_cushion', 'avg_yac_above_expectation'],
}

position_models_skill = {}

for pos, feats in position_features_skill.items():
    df_pos = season_features[season_features['position'] == pos].dropna(subset=feats + ['fantasy_points_ppr']).copy()
    X, y = df_pos[feats], df_pos['fantasy_points_ppr']

    model = RandomForestRegressor(random_state=SEED, n_estimators=300)
    kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
    cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
    model.fit(X, y)

    position_models_skill[pos] = model
    print(f"{pos}: {len(df_pos)} player-seasons, 10-fold CV R2 = {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

## Efficiency Model — RB

Opportunity (`rb_carry_share`, `rb_target_share`) reported directly. Efficiency modeled as `pts_per_touch` (rate stat) so volume can't dominate, with `receiving_touch_share` as an explicit role-mix feature (pass-catching back vs. bell-cow runner).

In [ ]:
# pull team onto base, build team-season totals for share metrics
base_team = base.merge(
    weekly_wnt[['player_id', 'season', 'week', 'team']].drop_duplicates(),
    on=['player_id', 'season', 'week'], how='left'
)

team_season_totals = base_team.groupby(['team', 'season']).agg(
    team_carries=('carries', 'sum'), team_targets=('targets', 'sum'), team_attempts=('attempts', 'sum'),
).reset_index()

season_features_team = base_team.groupby(
    ['player_id', 'player_display_name', 'position', 'season', 'team']
).agg(
    carries=('carries', 'sum'), targets=('targets', 'sum'), attempts=('attempts', 'sum'),
    rushing_epa=('rushing_epa', 'mean'), receiving_epa=('receiving_epa', 'mean'),
    rush_yards_over_expected_per_att=('rush_yards_over_expected_per_att', 'mean'),
    efficiency=('efficiency', 'mean'),
    percent_attempts_gte_eight_defenders=('percent_attempts_gte_eight_defenders', 'mean'),
    games_played=('week', 'nunique'), fantasy_points_ppr=('fantasy_points_ppr', 'sum'),
).reset_index()

season_features_team = season_features_team.merge(team_season_totals, on=['team', 'season'], how='left')

season_features_team['rb_carry_share'] = season_features_team['carries'] / season_features_team['team_carries']
season_features_team['rb_target_share'] = season_features_team['targets'] / season_features_team['team_targets']
season_features_team['rb_touches'] = season_features_team['carries'] + season_features_team['targets']
season_features_team['pts_per_touch'] = season_features_team['fantasy_points_ppr'] / season_features_team['rb_touches']
season_features_team['receiving_touch_share'] = season_features_team['targets'] / season_features_team['rb_touches']

season_features_team['qb_attempt_share'] = season_features_team['attempts'] / season_features_team['team_attempts']
season_features_team['qb_dropbacks_plus_carries'] = season_features_team['attempts'] + season_features_team['carries']
season_features_team['rush_share_of_offense'] = season_features_team['carries'] / season_features_team['qb_dropbacks_plus_carries']

print("season_features_team built:", season_features_team.shape)

In [ ]:
rb_efficiency_features_v3 = ['rushing_epa', 'receiving_epa', 'rush_yards_over_expected_per_att',
                              'efficiency', 'percent_attempts_gte_eight_defenders', 'receiving_touch_share']

df_rb_eff = season_features_team[
    (season_features_team['position'] == 'RB') &
    (season_features_team['games_played'] >= GAMES_FLOOR) &
    (season_features_team['rb_touches'] >= 20)
].dropna(subset=rb_efficiency_features_v3 + ['pts_per_touch']).copy()

X_rb = df_rb_eff[rb_efficiency_features_v3]
y_rb = df_rb_eff['pts_per_touch']

model_rb_v3 = RandomForestRegressor(random_state=SEED, n_estimators=300)
kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
cv_scores_rb = cross_val_score(model_rb_v3, X_rb, y_rb, cv=kf, scoring='r2')
model_rb_v3.fit(X_rb, y_rb)

print(f"RB efficiency model: {len(df_rb_eff)} player-seasons, 10-fold CV R2 = {cv_scores_rb.mean():.3f} (+/- {cv_scores_rb.std():.3f})")

## Efficiency Model — QB

Built on a 2021-2025 window (needed a larger sample than 2023-2025 alone for stable CV). Opportunity (`qb_attempt_share`) reported directly. Efficiency modeled as `pts_per_play` (points per dropback+carry, so rushing QB production doesn't skew the rate), with `rush_share_of_offense` as an explicit role-mix feature.

In [ ]:
QB_START_YEAR = 2021
QB_END_YEAR = 2025

weekly_qb_range = weekly_hist[
    (weekly_hist['season'] >= QB_START_YEAR) & (weekly_hist['season'] <= QB_END_YEAR)
].copy()

team_season_totals_qb = weekly_qb_range.groupby(['team', 'season']).agg(
    team_attempts=('attempts', 'sum')
).reset_index()

season_features_qb_range = weekly_qb_range.groupby(
    ['player_id', 'player_display_name', 'position', 'season', 'team']
).agg(
    attempts=('attempts', 'sum'), carries=('carries', 'sum'),
    passing_epa=('passing_epa', 'mean'), passing_cpoe=('passing_cpoe', 'mean'),
    rushing_epa=('rushing_epa', 'mean'), games_played=('week', 'nunique'),
    fantasy_points_ppr=('fantasy_points_ppr', 'sum'),
).reset_index()

season_features_qb_range = season_features_qb_range.merge(team_season_totals_qb, on=['team', 'season'], how='left')

# NGS passing must be pulled for the SAME range -- the original 2023-2025 pull doesn't cover 2021-2022
ngs_qb_range = nfl.load_nextgen_stats(seasons=list(range(QB_START_YEAR, QB_END_YEAR + 1)), stat_type='passing').to_pandas()
ngs_pass_range = ngs_qb_range.groupby(['player_gsis_id', 'season']).agg(
    avg_time_to_throw=('avg_time_to_throw', 'mean'),
    completion_percentage_above_expectation=('completion_percentage_above_expectation', 'mean'),
    aggressiveness=('aggressiveness', 'mean'),
).reset_index()

season_features_qb_range = season_features_qb_range.merge(
    ngs_pass_range, left_on=['player_id', 'season'], right_on=['player_gsis_id', 'season'], how='left'
).drop(columns='player_gsis_id')

season_features_qb_range['qb_dropbacks_plus_carries'] = season_features_qb_range['attempts'] + season_features_qb_range['carries']
season_features_qb_range['rush_share_of_offense'] = season_features_qb_range['carries'] / season_features_qb_range['qb_dropbacks_plus_carries']
season_features_qb_range['pts_per_play'] = season_features_qb_range['fantasy_points_ppr'] / season_features_qb_range['qb_dropbacks_plus_carries']
season_features_qb_range['qb_attempt_share'] = season_features_qb_range['attempts'] / season_features_qb_range['team_attempts']

print(f"season_features_qb_range: {season_features_qb_range.shape}")

In [ ]:
qb_efficiency_features_v5 = ['passing_epa', 'passing_cpoe', 'avg_time_to_throw',
                              'completion_percentage_above_expectation', 'aggressiveness',
                              'rushing_epa', 'rush_share_of_offense']

df_qb_v5_clean = season_features_qb_range[
    (season_features_qb_range['games_played'] >= GAMES_FLOOR) &
    (season_features_qb_range['attempts'] >= 100)
].dropna(subset=qb_efficiency_features_v5 + ['pts_per_play']).copy()

X_qb = df_qb_v5_clean[qb_efficiency_features_v5]
y_qb = df_qb_v5_clean['pts_per_play']

model_qb_v5 = RandomForestRegressor(random_state=SEED, n_estimators=300)
kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=SEED)
cv_scores_qb = cross_val_score(model_qb_v5, X_qb, y_qb, cv=kf, scoring='r2')
model_qb_v5.fit(X_qb, y_qb)

print(f"QB efficiency model: {len(df_qb_v5_clean)} player-seasons, 10-fold CV R2 = {cv_scores_qb.mean():.3f} (+/- {cv_scores_qb.std():.3f})")

## Combine — Dynasty Utility Score

`dynasty_utility = expected_season_points x retention_multiplier x (1 - exit_rate) x age_cliff_factor`

In [ ]:
RANK_SEASON = 2025

def attach_age(df, id_col='player_id'):
    latest = weekly_wnt[weekly_wnt['season'] == RANK_SEASON].sort_values('week').groupby('player_id').last()
    df = df.merge(latest['age_years'].rename('age_years'), left_on=id_col, right_index=True, how='left')
    df = df.merge(latest['age_WearNTear'].rename('age_WearNTear'), left_on=id_col, right_index=True, how='left')
    return df


def finalize_utility(df, pos):
    df = df.dropna(subset=['age_WearNTear']).copy()
    df['wnt_bucket'] = df['age_WearNTear'].apply(lambda x: assign_bucket(x, pos))
    df = df.merge(
        aging_curve_lookup[aging_curve_lookup['position'] == pos][['wnt_bucket', 'retention_multiplier', 'exit_rate']],
        on='wnt_bucket', how='left'
    )
    df['age_cliff_factor'] = df['age_years'].apply(age_cliff_factor)
    df['dynasty_utility'] = (
        df['expected_season_points'] * df['retention_multiplier']
        * (1 - df['exit_rate']) * df['age_cliff_factor']
    )
    df['position'] = pos
    return df[['player_display_name', 'position', 'season', 'expected_season_points', 'age_years',
               'wnt_bucket', 'retention_multiplier', 'exit_rate', 'age_cliff_factor', 'dynasty_utility']]


# QB
qb_util = df_qb_v5_clean[df_qb_v5_clean['season'] == RANK_SEASON].copy()
qb_util['expected_season_points'] = model_qb_v5.predict(qb_util[qb_efficiency_features_v5]) * qb_util['qb_dropbacks_plus_carries']
qb_util = attach_age(qb_util)

# RB
rb_util = df_rb_eff[df_rb_eff['season'] == RANK_SEASON].copy()
rb_util['expected_season_points'] = model_rb_v3.predict(rb_util[rb_efficiency_features_v3]) * rb_util['rb_touches']
rb_util = attach_age(rb_util)

# WR
wr_df = season_features[season_features['position'] == 'WR'].dropna(
    subset=position_features_skill['WR'] + ['fantasy_points_ppr']).copy()
wr_util = wr_df[wr_df['season'] == RANK_SEASON].copy()
wr_util['expected_season_points'] = position_models_skill['WR'].predict(wr_util[position_features_skill['WR']])
wr_util = attach_age(wr_util)

# TE
te_df = season_features[season_features['position'] == 'TE'].dropna(
    subset=position_features_skill['TE'] + ['fantasy_points_ppr']).copy()
te_util = te_df[te_df['season'] == RANK_SEASON].copy()
te_util['expected_season_points'] = position_models_skill['TE'].predict(te_util[position_features_skill['TE']])
te_util = attach_age(te_util)

qb_final = finalize_utility(qb_util, 'QB')
rb_final = finalize_utility(rb_util, 'RB')
wr_final = finalize_utility(wr_util, 'WR')
te_final = finalize_utility(te_util, 'TE')

print("Per-position utility tables built:",
      {p: len(d) for p, d in [('QB', qb_final), ('RB', rb_final), ('WR', wr_final), ('TE', te_final)]})

In [ ]:
combined = pd.concat([qb_final, rb_final, wr_final, te_final], ignore_index=True)
combined = combined.sort_values('dynasty_utility', ascending=False).reset_index(drop=True)
combined.insert(0, 'rank', combined.index + 1)

top_250 = combined.head(250)

print(f"Combined pool before cutoff: {len(combined)} players")
print("\nPosition breakdown within top 250:")
print(top_250['position'].value_counts())
print()
pd.set_option('display.max_rows', 250)
top_250

# Consensus Blend: Market Comparison, Divergence, and Final Blended Board

Everything below is a separate layer on top of the independent model above. Nothing here feeds
back into `base`, the feature tables, or any model training data -- see the note at the top of
each stage. Stages follow: (1) pull consensus independently, (2) align to `top_250` by player,
(3) validate the independent model against consensus *before* any blending, (4) surface
buy-low/sell-high divergence, (5) build an output-level blended score, (6) fold in positional
scarcity and flag the rookie-track-record gap.

In [ ]:
import requests
from scipy.stats import spearmanr
import re

print("Consensus-layer libraries loaded")

## Stage 1 -- Consensus Layer (FantasyCalc)

Pulled independently via FantasyCalc's public values endpoint (no official developer API, but
this endpoint is stable and widely used in the fantasy analytics community -- no auth key
required). Deliberately kept in its own DataFrames (`consensus_raw`, `consensus_df`) -- never
merged into `base`, any feature table, or model training data.

KeepTradeCut was also considered, but it has no stable public JSON endpoint (values are rendered
client-side); scraping it would be fragile and isn't worth the maintenance cost for this pass.
FantasyCalc alone is a reasonable single consensus source -- it's itself derived from ~1M+ real
trades, and RosterAudit/KTC-based aggregators (e.g. Dynatyze) confirm FantasyCalc is one of the
two boards the wider market treats as consensus. Swapping in a second source later is a matter of
writing another `fetch_*` function that returns the same `consensus_player_name` /
`consensus_position` / `consensus_value` shape.

In [ ]:
# --- Stage 1: pull consensus values, fully separate from the model pipeline ---
FANTASYCALC_URL = "https://api.fantasycalc.com/values/current"
FANTASYCALC_PARAMS = {
    "isDynasty": "true",
    "numQbs": 1,      # 1QB format -- change to 2 for superflex leagues
    "numTeams": 12,
    "ppr": 1,          # full PPR, matches fantasy_points_ppr target used throughout
}


def fetch_fantasycalc_values(url=FANTASYCALC_URL, params=FANTASYCALC_PARAMS) -> pd.DataFrame:
    """Pulls current dynasty trade values from FantasyCalc's public endpoint. Returns a
    standalone DataFrame -- never merged into `base`/feature tables/training data."""
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    raw = resp.json()

    rows = []
    for entry in raw:
        p = entry.get("player", {})
        rows.append({
            "consensus_player_name": p.get("name"),
            "consensus_position": p.get("position"),
            "consensus_team": p.get("maybeTeam"),
            "consensus_age": p.get("maybeAge"),
            "sleeper_id": p.get("sleeperId"),
            "consensus_value": entry.get("value"),
            "consensus_overall_rank": entry.get("overallRank"),
            "consensus_position_rank": entry.get("positionRank"),
            "consensus_trend_30day": entry.get("trend30Day"),
        })
    return pd.DataFrame(rows)


consensus_raw = fetch_fantasycalc_values()
print(f"Consensus values pulled: {consensus_raw.shape}")
consensus_raw.head()

In [ ]:
# normalize to rank / percentile within position -- QB/RB/WR/TE only, matching the model's scope
consensus_df = consensus_raw[consensus_raw["consensus_position"].isin(["QB", "RB", "WR", "TE"])].copy()
consensus_df = consensus_df.sort_values("consensus_value", ascending=False).reset_index(drop=True)

consensus_df["consensus_position_percentile"] = (
    consensus_df.groupby("consensus_position")["consensus_value"].rank(pct=True) * 100
)

print(f"consensus_df (QB/RB/WR/TE only): {consensus_df.shape}")
consensus_df.head()

## Stage 2 -- Align `top_250` to Consensus by Player

Name matching only (no shared player ID between my pipeline's `gsis_id` world and FantasyCalc's
`sleeperId` world without an extra crosswalk), so names are normalized -- lowercased, punctuation
stripped, suffixes (Jr./Sr./II/III/IV) dropped -- and joined with position as a tiebreaker. An
outer join keeps and flags players present in only one source: day-3 rookies my model may not
have a qualifying NFL sample for, or players consensus boards haven't priced yet.

In [ ]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}


def normalize_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    name = re.sub(r"[.'\-]", "", name)
    tokens = [t for t in name.split() if t not in SUFFIXES]
    return " ".join(tokens)


top_250 = top_250.copy()
top_250["match_key"] = top_250["player_display_name"].apply(normalize_name) + "|" + top_250["position"]
consensus_df["match_key"] = consensus_df["consensus_player_name"].apply(normalize_name) + "|" + consensus_df["consensus_position"]

aligned = top_250.merge(consensus_df, on="match_key", how="outer", indicator=True)

model_only = aligned[aligned["_merge"] == "left_only"].copy()
consensus_only = aligned[aligned["_merge"] == "right_only"].copy()
matched = aligned[aligned["_merge"] == "both"].copy()

print(f"Matched (in both sources): {len(matched)}")
print(f"Model-only (my top 250, no consensus match -- e.g. limited-sample rookies): {len(model_only)}")
print(f"Consensus-only (priced by the market, outside my top 250 / not modeled): {len(consensus_only)}")

## Stage 3 -- Validate Independently (clean benchmark, before any blending)

Spearman rank correlation between `dynasty_utility` rank and FantasyCalc consensus rank, computed
only on the `matched` player pool, with nothing from consensus fed into the model beforehand.
This is the number that goes in the README / resume as "how well does the independent model track
the market" -- it has to be computed before Stage 5's blend exists, or it stops meaning anything.

In [ ]:
# --- Stage 3: independent validation -- run this BEFORE Stage 5 exists in any kernel session ---
matched["model_rank"] = matched["rank"]
matched["consensus_rank_for_corr"] = matched["consensus_overall_rank"]

overall_corr, overall_p = spearmanr(matched["model_rank"], matched["consensus_rank_for_corr"])
print(f"Overall Spearman rank correlation (model vs. consensus, n={len(matched)}): "
      f"{overall_corr:.3f} (p={overall_p:.4f})")

position_corr = {}
for pos in ["QB", "RB", "WR", "TE"]:
    sub = matched[matched["position"] == pos]
    if len(sub) >= 5:
        rho, p = spearmanr(sub["model_rank"], sub["consensus_rank_for_corr"])
        position_corr[pos] = {"rho": round(rho, 3), "p": round(p, 4), "n": len(sub)}

position_corr_df = pd.DataFrame(position_corr).T
print("\nBy-position Spearman correlation:")
position_corr_df

## Stage 4 -- Divergence Analysis (the actual insight layer)

`rank_divergence = consensus_rank - model_rank`. Positive means my model ranks the player better
than the market does (buy-low candidate); negative means the market ranks the player better than
my model does (sell-high candidate). Per the README framing in Stage 7: this is "model and market
disagree," not "model is right and market is wrong" -- the model scores current skill x
aging/retention, the market blends in scouting/narrative/injury information the model doesn't
see.

In [ ]:
# --- Stage 4: buy-low / sell-high divergence table ---
matched["rank_divergence"] = matched["consensus_rank_for_corr"] - matched["model_rank"]

buy_low = matched.sort_values("rank_divergence", ascending=False).head(20)[
    ["player_display_name", "position", "model_rank", "consensus_rank_for_corr", "rank_divergence"]
].reset_index(drop=True)

sell_high = matched.sort_values("rank_divergence", ascending=True).head(20)[
    ["player_display_name", "position", "model_rank", "consensus_rank_for_corr", "rank_divergence"]
].reset_index(drop=True)

print("Top 20 buy-low (model likes more than market):")
display(buy_low)
print("\nTop 20 sell-high (market likes more than model):")
display(sell_high)

## Stage 5 -- Output-Level Blend

`final_dynasty_value = alpha * model_utility_percentile + (1 - alpha) * consensus_percentile`.
Built only from the two percentile columns computed above -- consensus never touches the model's
training data, so Stage 3's validation and Stage 4's divergence table both stay meaningful after
this cell runs.

`alpha` is context-dependent rather than a flat constant: it leans toward the independent model
for players with a real multi-season NFL statistical track record, and toward consensus for
players with little or no qualifying NFL sample, where the market is pricing in draft
capital/scouting information the stat model can't see yet (this is also where Stage 6's rookie gap
shows up most directly).

In [ ]:
# --- Stage 5: output-level blend, built from percentiles only -- no consensus in model inputs ---
matched["model_utility_percentile"] = matched.groupby("position")["dynasty_utility"].rank(pct=True) * 100
# consensus_position_percentile already computed in Stage 1

# NOTE: top_250 / matched never carried player_id -- finalize_utility() (Combine stage above)
# only keeps player_display_name, so seasons_seen has to join on name + position instead.
seasons_seen = season_agg_hist_sorted.groupby(
    ["player_display_name", "position"]
)["season"].nunique().rename("seasons_seen").reset_index()
matched = matched.merge(seasons_seen, on=["player_display_name", "position"], how="left")
matched["seasons_seen"] = matched["seasons_seen"].fillna(0)


def alpha_for(seasons):
    """Trust the independent model more as NFL sample size grows; lean on consensus when it's thin."""
    if seasons >= 3:
        return 0.75
    elif seasons == 2:
        return 0.6
    elif seasons == 1:
        return 0.45
    else:
        return 0.3  # rookie / no qualifying NFL sample


matched["alpha"] = matched["seasons_seen"].apply(alpha_for)
matched["final_dynasty_value"] = (
    matched["alpha"] * matched["model_utility_percentile"]
    + (1 - matched["alpha"]) * matched["consensus_position_percentile"]
)

final_board = matched.sort_values("final_dynasty_value", ascending=False).reset_index(drop=True)
final_board.insert(0, "final_rank", final_board.index + 1)

print(f"Final blended board: {len(final_board)} players")
final_board[["final_rank", "player_display_name", "position", "model_rank",
             "consensus_rank_for_corr", "alpha", "final_dynasty_value"]].head(25)

## Stage 6a -- Known Gap: Positional Scarcity (Replacement-Level Adjustment)

`dynasty_utility` as built scores each position on its own curve, so a QB6 and a WR30 aren't
directly comparable -- there's no cross-position replacement-level normalization. This subtracts a
per-position replacement-level baseline (standard 12-team, 1QB startable-player cutoffs) so the
adjusted score is comparable across positions. Folded in as an alternate ranking
(`dynasty_utility_vbd`) on `combined` rather than replacing `dynasty_utility`, so the original
per-position score used in Stages 3-5 above stays untouched.

In [ ]:
# --- Stage 6a: positional scarcity via replacement-level (VBD-style) adjustment ---
REPLACEMENT_RANK = {"QB": 12, "RB": 30, "WR": 36, "TE": 12}


def replacement_level(df, pos, value_col="dynasty_utility"):
    pos_sorted = df[df["position"] == pos].sort_values(value_col, ascending=False)
    cutoff = REPLACEMENT_RANK[pos]
    if len(pos_sorted) >= cutoff:
        return pos_sorted.iloc[cutoff - 1][value_col]
    return pos_sorted[value_col].min() if len(pos_sorted) else 0.0


replacement_levels = {pos: replacement_level(combined, pos) for pos in REPLACEMENT_RANK}
print("Replacement-level dynasty_utility by position:", replacement_levels)

combined["dynasty_utility_vbd"] = combined.apply(
    lambda r: r["dynasty_utility"] - replacement_levels[r["position"]], axis=1
)
combined_scarcity = combined.sort_values("dynasty_utility_vbd", ascending=False).reset_index(drop=True)
combined_scarcity.insert(0, "scarcity_rank", combined_scarcity.index + 1)

print("\nTop 15 cross-position, scarcity-adjusted:")
combined_scarcity[["scarcity_rank", "player_display_name", "position",
                    "dynasty_utility", "dynasty_utility_vbd"]].head(15)

## Stage 6b -- Known Gap: Rookies Without an NFL Track Record (placeholder / follow-up)

Rookies with zero qualifying NFL seasons fall out of the skill models entirely, since those models
are trained on weekly NFL production -- there's nothing for them to predict from yet. The Stage 5
blend already leans on consensus (`alpha = 0.3`) for these players as an interim fix, since draft
boards price in scouting/college-production signal the stat model can't see.

**Not built here, left as a follow-up iteration:** pull draft capital (round/pick, via
`nfl.load_draft_picks()`) and college production (target share, breakout age, etc. -- would need a
separate college stats source) as features, fit a small separate "rookie prior" model, and use its
output in place of `expected_season_points` for any player with `seasons_seen == 0`, ahead of the
Stage 5 blend. Flagged below so the gap is visible rather than silently absorbed into the alpha
fallback.

In [ ]:
# --- Stage 6b: flag the players currently relying on the alpha=0.3 fallback ---
rookies_no_track_record = matched[matched["seasons_seen"] == 0][
    ["player_display_name", "position", "model_rank", "consensus_rank_for_corr", "alpha"]
].reset_index(drop=True)

print(f"Players relying on the alpha=0.3 fallback (no qualifying NFL track record): "
      f"{len(rookies_no_track_record)}")
rookies_no_track_record.head(15)